## <a href="https://www.kaggle.com/competitions/isic-2024-challenge">ISIC 2024</a> baseline

## Created by <a href="https://github.com/yunsuxiaozi/">yunsuxiaozi </a> 2024/09/03

## Attention:This competition will end in 4 days, and this should be the final version updated during the competition. Trust your CV and good luck to everyone.

- v1:0.165

- v2:0.166

- v3:0.165

- v4:downsample:0.165

- v5:0.165

- v6:no downsample :0.164

- v7:downsample 100000: 0.164

- v8:groupkfold：0.164

- v9:500 iterations:0.165

- v10:EDA FeatureEngineer:0.169

- v11:GPU+fusion_weight:0.172

- v12:optuna + lgb_params:0.173

- v13:optuna + xgb_params: 0.170

- v14: groupby age_approx+diff_feature+optuna cat_params: 0.175

- v15: add groupby age_approx feat+mean_fusion_weight: 0.158

- v16:CV:0.16707820664344494 LB:0.175

- v17:downsample 90000+new_cat_params:LB:0.176

- v18:downsample 10000:0.177

- v19:groupanatom_site_general_diff feats +new_lgb_params :CV>=0.17022408072239104,LB=0.177

- v20:failed

- v21:downsample 9000+fusion_weight:CV>=0.16992930943007875,LB=0.177

- v22:downsample 10000:CV>=0.17022408072239104,LB=0.178

- v23: ['age_approx','anatom_site_general'] + category sum:CV>=0.17119463650274197,LB=0.179

- v24:nunique_cols + drop_cols +new_lgb_params: CV>=0.17012719242308885,LB=0.179

- v25: std_feats:CV>=0.17006167682462642,LB=0.179

- v26:diff2:CV>=0.168691127999616,LB=0.181

- v27:logit+upsample:CV>=0.1675851037539037,LB=0.182

- v28:new_lgb_params + diff2-diff1 + faster_transformer:CV=0.1694763054120487,LB=0.183

- v29:3 sampling strategies+minmax probability normalization:

## 1.Libraries and Config

In [ ]:
import pandas as pd#导入csv文件的库
import numpy as np#矩阵运算与科学计算的库
#model lgb 分类模型,日志评估  3大树模型
from  lightgbm import LGBMClassifier,log_evaluation
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from tqdm import tqdm#显示进度条的库
import scipy#在numpy的基础上提供了广泛的科学计算功能
#metric
from sklearn.metrics import roc_auc_score#导入roc_auc曲线
#KFold是直接分成k折,StratifiedKFold还要考虑每种类别的占比
from sklearn.model_selection import GroupKFold
import warnings#避免一些可以忽略的报错
warnings.filterwarnings('ignore')#filterwarnings()方法是用于设置警告过滤器的方法，它可以控制警告信息的输出方式和级别。

#config
class Config():
    seed=2024#随机种子
    num_folds=10#K折交叉验证
    TARGET_NAME ='target'#标签
import random#提供了一些用于生成随机数的函数
#设置随机种子,保证模型可以复现
def seed_everything(seed):
    np.random.seed(seed)#numpy的随机种子
    random.seed(seed)#python内置的随机种子
seed_everything(Config.seed)

## 2.Read Dataset

We can see that the ratio of targets is 1000:1.

In [ ]:
train=pd.read_csv("/kaggle/input/isic-2024-challenge/train-metadata.csv")
print(f"len(train):{len(train)}")
test=pd.read_csv("/kaggle/input/isic-2024-challenge/test-metadata.csv")
print(f"len(test):{len(test)}")

# # 线下节省GPU时间
# if len(test)==3:
#     train=train[:40000]

train.head()

In [ ]:
#40W和400
train[Config.TARGET_NAME].value_counts()

##  3.My EDA conclusion in Chinese.

In [ ]:
#表格型数据的EDA分析结果:
#isic_id是唯一的,patient_id为1042个,target是label
#sex:男性的阳性率比女性高一点5:4
#age_approx:18岁以下没有患病,age和target.mean()呈现较高的皮尔逊相关系数
#anatom_site_general:在head/neck的时候有64,其余类别都在8左右
#clin_size_long_diam_mm:观察count呈现长尾分布,有些地方target.mean()=0也许是样本数量太少,不能主观臆断
#image_type就一个类别,没什么用
#tbp_tile_type:3D:XP和3D:white target.mean() 6:17
#tbp_lv_A:连续值,服从正态分布
#tbp_lv_Aext:连续值,服从正态分布
#tbp_lv_B:连续值,服从正态分布
#tbp_lv_Bext:连续值,服从正态分布
#tbp_lv_C:连续值,服从正态分布
#tbp_lv_Cext:连续值,服从正态分布
#tbp_lv_H:连续值,服从正态分布
#tbp_lv_Hext:连续值,服从正态分布
#tbp_lv_L:连续值,服从正态分布
#tbp_lv_Lext:连续值,服从正态分布
#tbp_lv_areaMM2:数据呈现长尾分布
#tbp_lv_area_perim_ratio:数据呈现长尾分布
#tbp_lv_color_std_mean:超级长尾分布,可以考虑按照是不是0来分成2类
#tbp_lv_deltaA:连续值,服从正态分布
#tbp_lv_deltaB:连续值,服从正态分布
#tbp_lv_deltaL:连续值,服从正态分布
#tbp_lv_deltaLB:连续值,服从正态分布,大于等于25的可能是异常值
#tbp_lv_deltaLBnorm:连续值,服从正态分布,大于等于20的可能是异常值
#tbp_lv_eccentricity:连续值,服从正态分布
#tbp_lv_location:Head & Neck好像是最有病的,另外有几个类别target.mean()=0,不确定是数据太少还是可以当做结论来用
#tbp_lv_location_simple:类别可能和前面有重复的
#tbp_lv_minorAxisMM:长尾分布
#tbp_lv_nevi_confidence:右偏的长尾分布
#tbp_lv_norm_border:右偏的长尾分布
#tbp_lv_norm_color:在10和0有大量聚集,考虑0,10,other3个类别
#tbp_lv_perimeterMM:长尾分布
#tbp_lv_radial_color_std_max:0是大多数的长尾分布
#tbp_lv_stdL:有点长尾分布
#tbp_lv_stdLExt:长尾分布
#tbp_lv_symm_2axis:连续值,在几个点count特别多
#tbp_lv_symm_2axis_angle:类别型变量
#tbp_lv_x:正态分布连续值
#tbp_lv_y:左偏正态分布连续值
#tbp_lv_z:连续值,服从正态分布
#attribution:类别型变量
#copyright_license:CC-BY-NC的均值特别低
#lesion_id:2万个,40万训练数据
#iddx_full:39万多都是1个类别(Benign),剩下的类别还用细分吗
#iddx_1:3个类别的类别型变量
#iddx_2:有缺失值和没缺失值可以1个类别
#iddx_3:有缺失值和没缺失值可以1个类别
#iddx_4:有缺失值和没缺失值可以1个类别
#iddx_5就1个数值,其余全是缺失值,drop掉吧
#mel_mitotic_index:少数有值,类别型变量还有大小关系
#mel_thick_mm:有值和没值搞一个类别,就只有少数有值
#tbp_lv_dnn_lesion_confidence:连续型变量,右偏长尾分布

## 4.Feature Engineer

Part of the features were created by myself, and part came from <a href="https://www.kaggle.com/code/abdmental01/multimodel-isic/notebook">multimodel-isic</a>.

In [ ]:
#训练数据里的类别型变量
print("tbp_lv_dnn_lesion_confidence feature")
cates=['age_approx', 'sex', 'anatom_site_general', 'tbp_tile_type', 'tbp_lv_location_simple', 'attribution', 'copyright_license']
#'tbp_lv_dnn_lesion_confidence'构造groupby特征
cates2mean={}
for c in cates:
    base=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].mean().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'mean_{c}_tbp_lv_dnn_lesion_confidence'})
    
    min_tmp=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].min().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'min_{c}_tbp_lv_dnn_lesion_confidence'})
    base=base.merge(min_tmp,on=c,how='left')
    
    max_tmp=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].max().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'max_{c}_tbp_lv_dnn_lesion_confidence'})
    base=base.merge(max_tmp,on=c,how='left')
    
    median_tmp=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].median().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'median_{c}_tbp_lv_dnn_lesion_confidence'})
    base=base.merge(median_tmp,on=c,how='left')
    
    std_tmp=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].std().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'std_{c}_tbp_lv_dnn_lesion_confidence'})
    base=base.merge(std_tmp,on=c,how='left')
    
    skew_tmp=train.groupby(c)['tbp_lv_dnn_lesion_confidence'].skew().reset_index().rename(columns={'tbp_lv_dnn_lesion_confidence':f'skew_{c}_tbp_lv_dnn_lesion_confidence'})
    base=base.merge(skew_tmp,on=c,how='left')
    
    cates2mean[c]=base

def FE(df):
    #特征工程出处:https://www.kaggle.com/code/abdmental01/multimodel-isic
    #这部分特征工程可能需要一些专业的背景知识,这里我也没有深究 
    df["lesion_size_ratio"]=df["tbp_lv_minorAxisMM"]/df["clin_size_long_diam_mm"]
    df["lesion_shape_index"]=df["tbp_lv_areaMM2"]/(df["tbp_lv_perimeterMM"]**2)
    df["hue_contrast"]= (df["tbp_lv_H"]-df["tbp_lv_Hext"]).abs()
    df["luminance_contrast"]= (df["tbp_lv_L"]-df["tbp_lv_Lext"]).abs()
    df["lesion_color_difference"]=np.sqrt(df["tbp_lv_deltaA"]**2+df["tbp_lv_deltaB"]**2+df["tbp_lv_deltaL"]**2)
    df["border_complexity"]=df["tbp_lv_norm_border"]+df["tbp_lv_symm_2axis"]
    df["3d_position_distance"]=np.sqrt(df["tbp_lv_x"]**2+df["tbp_lv_y"]**2+df["tbp_lv_z"]**2)
    df["perimeter_to_area_ratio"]=df["tbp_lv_perimeterMM"]/df["tbp_lv_areaMM2"]
    df["lesion_visibility_score"]=df["tbp_lv_deltaLBnorm"]+df["tbp_lv_norm_color"]
    df["combined_anatomical_site"]=df["anatom_site_general"]+"_"+df["tbp_lv_location"]
    df["symmetry_border_consistency"]=df["tbp_lv_symm_2axis"]*df["tbp_lv_norm_border"]
    df["color_consistency"]=df["tbp_lv_stdL"]/df["tbp_lv_Lext"]
    df["size_age_interaction"]=df["clin_size_long_diam_mm"]*df["age_approx"]
    df["hue_color_std_interaction"]=df["tbp_lv_H"]*df["tbp_lv_color_std_mean"]
    df["lesion_severity_index"]=(df["tbp_lv_norm_border"]+df["tbp_lv_norm_color"]+df["tbp_lv_eccentricity"])/3
    df["shape_complexity_index"]=df["border_complexity"]+df["lesion_shape_index"]
    df["color_contrast_index"]=df["tbp_lv_deltaA"]+df["tbp_lv_deltaB"]+df["tbp_lv_deltaL"]+df["tbp_lv_deltaLBnorm"]
    df["normalized_lesion_size"]=df["clin_size_long_diam_mm"]/df["age_approx"]
    df["mean_hue_difference"]=(df["tbp_lv_H"]+df["tbp_lv_Hext"])/2
    df["std_dev_contrast"]=np.sqrt((df["tbp_lv_deltaA"]**2+df["tbp_lv_deltaB"]**2+df["tbp_lv_deltaL"]**2)/3)
    df["color_shape_composite_index"]=(df["tbp_lv_color_std_mean"]+df["tbp_lv_area_perim_ratio"]+df["tbp_lv_symm_2axis"])/3
    df["3d_lesion_orientation"]=np.arctan2(df["tbp_lv_y"],df["tbp_lv_x"])
    df["overall_color_difference"]=(df["tbp_lv_deltaA"]+df["tbp_lv_deltaB"]+df["tbp_lv_deltaL"])/3
    df["symmetry_perimeter_interaction"]=df["tbp_lv_symm_2axis"]*df["tbp_lv_perimeterMM"]
    df["comprehensive_lesion_index"]=(df["tbp_lv_area_perim_ratio"]+df["tbp_lv_eccentricity"]+df["tbp_lv_norm_color"]+df["tbp_lv_symm_2axis"])/4

    
    print("drop_cols")
    drop_cols=['lesion_id',#训练数据有且测试数据没有,缺失值占比0.945,每个id出现1次,故drop
     'iddx_2', #训练数据有且测试数据没有,缺失值占比0.997,故drop
     'iddx_3', #训练数据有且测试数据没有,缺失值占比0.997,故drop
     'iddx_4',#训练数据有且测试数据没有,缺失值占比0.998,故drop
     'iddx_5',#训练数据有且测试数据没有,缺失值占比0.99999,故drop
     'mel_mitotic_index',#训练数据有且测试数据没有,缺失值占比0.9998,故drop
     'mel_thick_mm',#训练数据有且测试数据没有,缺失值占比0.9998,故drop    
     'image_type',#训练数据中nunique=1
     #'isic_id',#就像普通的id一样没什么意义
     #可能本来就是知道target才有这两列数据
     'iddx_full',#训练数据有且测试数据没有,和target有一一对应关系,每个类别target.mean()不是0就是1
     'iddx_1',#训练数据有且测试数据没有,和target有一一对应关系,每个类别target.mean()不是0就是1 
    ]
    #如果测试数据没有这些列可以忽略掉
    df.drop(drop_cols,axis=1,inplace=True,errors='ignore')
    print("tbp_lv_dnn_lesion_confidence feature")
    for c in cates:
        df=df.merge(cates2mean[c],on=c,how='left')
        
    print("age_approx feature")
    #年龄低于15岁的变成15岁
    df.loc[df['age_approx']<=15,'age_approx']=15
    #缺失值用最多的填充
    df.loc[(df['age_approx']!=df['age_approx']),'age_approx']=55
    value_counts={55.0: 58123,
                 65.0: 54946,
                 60.0: 54109,
                 50.0: 47924,
                 70.0: 39775,
                 40.0: 31297,
                 75.0: 30801,
                 45.0: 23580,
                 80.0: 21096,
                 35.0: 11543,
                 30.0: 10400,
                 85.0: 8847,
                 25.0: 3433,
                 20.0: 1742,
                 15.0: 645}
    df['age_approx_count']=df['age_approx'].apply(lambda x:value_counts.get(x,645))
    
    
    print("sex feature")
    #sex
    df['sex_male']=(df['sex']=='male').astype(np.int8)
    df['sex_female']=(df['sex']=='female').astype(np.int8)
    value_counts={'male':265546,'female':123996}
    #nan的value_counts
    df['sex']=df['sex'].apply(lambda x:value_counts.get(x,11517))
    
    #'anatom_site_general'  one-hot
    print("anatom_site_general feature")
    cols=['posterior torso','lower extremity','anterior torso','upper extremity','head/neck']
    for col in cols:
        df[f'anatom_site_general_{col}']=(df['anatom_site_general']==col).astype(np.int8)
    #value_counts
    value_counts={'posterior torso': 121902,
     'lower extremity': 103028,
     'anterior torso': 87770,
     'upper extremity': 70557,
     'head/neck': 12046}
    df['anatom_site_general']=df['anatom_site_general'].apply(lambda x:value_counts.get(x,12046))
    
    print("clin_size_long_diam_mm feature")
    #这个长尾分布感觉修正也修正的一般
    df['clin_size_long_diam_mm']=np.log1p(df['clin_size_long_diam_mm'])
    
    print("tbp_tile_type feature")
    df['tbp_tile_type']=(df['tbp_tile_type']=='3D: XP').astype(np.int8)
    
    print("tbp_lv_XX', 'tbp_lv_XXext feature")
    #不知道具体含义的暴力特征构造
    for c in ['A','B','C','H','L']:
        col1,col2=f'tbp_lv_{c}',f'tbp_lv_{c}ext'
        df[f'{col1}+{col2}']=df[col1]+df[col2]
        df[f'{col1}-{col2}']=df[col1]-df[col2]
        df[f'{col1}*{col2}']=df[col1]*df[col2]
        df[f'{col1}/{col2}']=df[col1]/(df[col2]+1e-20)  
        
    print("tbp_lv_areaMM2 feature")
    df['tbp_lv_areaMM2']=np.log1p(df['tbp_lv_areaMM2'])
    
    print("tbp_lv_area_perim_ratio feature")
    #tbp_lv_area_perim_ratio是长尾分布
    df['tbp_lv_area_perim_ratio']=np.log1p(df['tbp_lv_area_perim_ratio'])
    #修正大于4的异常值为均值
    df.loc[df['tbp_lv_area_perim_ratio']>=4,'tbp_lv_area_perim_ratio']=2.9
    
    print("tbp_lv_symm_2axis feature")
    #tbp_lv_symm_2axis_angle应该是一个角度,故考虑sin和cos
    df['sin_tbp_lv_symm_2axis_angle']=np.sin(2*np.pi*df['tbp_lv_symm_2axis_angle']/180)
    df['cos_tbp_lv_symm_2axis_angle']=np.cos(2*np.pi*df['tbp_lv_symm_2axis_angle']/180)
    df['tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle']=df['tbp_lv_symm_2axis']*df['sin_tbp_lv_symm_2axis_angle']
    df['tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle']=df['tbp_lv_symm_2axis']*df['cos_tbp_lv_symm_2axis_angle']
    df['tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle']=df['tbp_lv_symm_2axis']/df['sin_tbp_lv_symm_2axis_angle']
    df['tbp_lv_symm_2axis/cos_tbp_lv_symm_2axis_angle']=df['tbp_lv_symm_2axis']/df['cos_tbp_lv_symm_2axis_angle']
    
    print("tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z feature")
    #x,y,z也许是长方体的长宽高?用求体积和表面积的特征构造方法试试
    df['V_tbp_lv']=abs(df['tbp_lv_x']*df['tbp_lv_y']*df['tbp_lv_z'])
    df['S_tbp_lv']=2*(abs(df['tbp_lv_x']*df['tbp_lv_y'])+abs(df['tbp_lv_x']*df['tbp_lv_z'])+abs(df['tbp_lv_y']*df['tbp_lv_z']))
    
    print("copyright feature")
    cols=['CC-BY','CC-BY-NC','CC-0']
    for col in cols:
        df[f'copyright_license_{col}']=(df['copyright_license']==col).astype(np.int8)
    value_counts={'CC-BY': 188812, 'CC-BY-NC': 183582, 'CC-0': 28665}
    df['copyright_license']=df['copyright_license'].apply(lambda x:value_counts.get(x,28665))
    
    print("attribution feature")
    value_counts={'Memorial Sloan Kettering Cancer Center': 129068,
         'Department of Dermatology, Hospital Clínic de Barcelona': 105724,
         'University Hospital of Basel': 65218,
         'Frazer Institute, The University of Queensland, Dermatology Research Centre': 51768,
         'ACEMID MIA': 28665,
         'ViDIR Group, Department of Dermatology, Medical University of Vienna': 12640,
         'Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris': 7976}
    for key,value in value_counts.items():
        df[f"attribution_{key}"]=(df['attribution']==key).astype(np.int8)
    df['attribution']=df['attribution'].apply(lambda x:value_counts.get(x,7976))
    
    print("tbp_lv_location feature")
    value_counts={'Torso Back Top Third': 71112,
     'Torso Front Top Half': 63350,
     'Torso Back Middle Third': 46185,
     'Left Leg - Lower': 27428,
     'Right Leg - Lower': 25208,
     'Torso Front Bottom Half': 24360,
     'Left Leg - Upper': 23673,
     'Right Leg - Upper': 23034,
     'Right Arm - Upper': 22972,
     'Left Arm - Upper': 22816,
     'Head & Neck': 12046,
     'Left Arm - Lower': 11939,
     'Right Arm - Lower': 10636,
     'Unknown': 5756,
     'Torso Back Bottom Third': 4596,
     'Left Leg': 1974,
     'Right Leg': 1711,
     'Left Arm': 1593,
     'Right Arm': 601,
     'Torso Front': 60,
     'Torso Back': 9}
    for key,value in value_counts.items():
        df[f"tbp_lv_location_{key}"]=(df['tbp_lv_location']==key).astype(np.int8)
    #训练集没有出现过的key假设为训练集里最少的value
    df['tbp_lv_location']=df['tbp_lv_location'].apply(lambda x:value_counts.get(x,9))
    
    value_counts={'Torso Back': 121902,
     'Torso Front': 87770,
     'Left Leg': 53075,
     'Right Leg': 49953,
     'Left Arm': 36348,
     'Right Arm': 34209,
     'Head & Neck': 12046,
     'Unknown': 5756}    
    for key,value in value_counts.items():
        df[f"tbp_lv_location_simple_{key}"]=(df['tbp_lv_location_simple']==key).astype(np.int8)
    #训练集没有出现过的key假设为训练集里最少的value
    df['tbp_lv_location_simple']=df['tbp_lv_location_simple'].apply(lambda x:value_counts.get(x,5756))
    
    print("group feature")
    #tbp_lv_dnn_lesion_confidence
    float_cols=['clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A/tbp_lv_Aext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C-tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H-tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/cos_tbp_lv_symm_2axis_angle', 'V_tbp_lv', 'S_tbp_lv']
    for col in float_cols:
        df[f"mean_patient_id_{col}"]=df.groupby('patient_id')[col].transform('mean')
        df[f"min_patient_id_{col}"]=df.groupby('patient_id')[col].transform('min')
        df[f"max_patient_id_{col}"]=df.groupby('patient_id')[col].transform('max')
        df[f"std_patient_id_{col}"]=df.groupby('patient_id')[col].transform('std')
        df[f"median_patient_id_{col}"]=df.groupby('patient_id')[col].transform('median')
        df[f"skew_patient_id_{col}"]=df.groupby('patient_id')[col].transform('skew')

        df[f"{col}-mean_patient_id_{col}/std_patient_id_{col}"]=(df[col]-df[f"mean_patient_id_{col}"])/(df[f'std_patient_id_{col}']+1e-15)
        df[f"{col}/mean_patient_id_{col}"]=df[col]/(df[f"mean_patient_id_{col}"]+1e-15)
        df[f'ptp_patient_id_{col}']=df[f"max_patient_id_{col}"]-df[f"min_patient_id_{col}"]
    
    nunique_cols=['anatom_site_general', 'tbp_tile_type', 'tbp_lv_location_simple', 'attribution', 'copyright_license', 'max_age_approx_tbp_lv_dnn_lesion_confidence', 'mean_sex_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'max_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'mean_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'min_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'max_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'min_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'mean_attribution_tbp_lv_dnn_lesion_confidence', 'min_attribution_tbp_lv_dnn_lesion_confidence', 'max_attribution_tbp_lv_dnn_lesion_confidence', 'median_attribution_tbp_lv_dnn_lesion_confidence', 'std_attribution_tbp_lv_dnn_lesion_confidence', 'skew_attribution_tbp_lv_dnn_lesion_confidence', 'mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'min_copyright_license_tbp_lv_dnn_lesion_confidence', 'max_copyright_license_tbp_lv_dnn_lesion_confidence', 'median_copyright_license_tbp_lv_dnn_lesion_confidence', 'std_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'anatom_site_general_posterior torso', 'anatom_site_general_lower extremity', 'anatom_site_general_anterior torso', 'anatom_site_general_upper extremity', 'anatom_site_general_head/neck', 'copyright_license_CC-BY', 'copyright_license_CC-BY-NC', 'copyright_license_CC-0', 'attribution_Memorial Sloan Kettering Cancer Center', 'attribution_Department of Dermatology, Hospital Clínic de Barcelona', 'attribution_University Hospital of Basel', 'attribution_Frazer Institute, The University of Queensland, Dermatology Research Centre', 'attribution_ACEMID MIA', 'attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'attribution_Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris', 'tbp_lv_location_Torso Back Top Third', 'tbp_lv_location_Torso Front Top Half', 'tbp_lv_location_Torso Back Middle Third', 'tbp_lv_location_Left Leg - Lower', 'tbp_lv_location_Right Leg - Lower', 'tbp_lv_location_Torso Front Bottom Half', 'tbp_lv_location_Left Leg - Upper', 'tbp_lv_location_Right Leg - Upper', 'tbp_lv_location_Right Arm - Upper', 'tbp_lv_location_Left Arm - Upper', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_Left Arm - Lower', 'tbp_lv_location_Right Arm - Lower', 'tbp_lv_location_Unknown', 'tbp_lv_location_Torso Back Bottom Third', 'tbp_lv_location_Left Leg', 'tbp_lv_location_Right Leg', 'tbp_lv_location_Left Arm', 'tbp_lv_location_Right Arm', 'tbp_lv_location_Torso Front', 'tbp_lv_location_Torso Back', 'tbp_lv_location_simple_Torso Back', 'tbp_lv_location_simple_Torso Front', 'tbp_lv_location_simple_Left Leg', 'tbp_lv_location_simple_Right Leg', 'tbp_lv_location_simple_Left Arm', 'tbp_lv_location_simple_Right Arm', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_location_simple_Unknown']
    for col in nunique_cols:
        df[f"sum_patient_id_{col}"]=df.groupby('patient_id')[col].transform('sum')
        df[f"skew_patient_id_{col}"]=df.groupby('patient_id')[col].transform('skew')
        df[f"std_patient_id_{col}"]=df.groupby('patient_id')[col].transform('std')
        
        
    #patient_id的count特征
    tmp=df.groupby('patient_id')['tbp_lv_B'].count().reset_index().rename(columns={"tbp_lv_B":"tbp_lv_B_count"})
    df=df.merge(tmp,on='patient_id',how='left')
    
    print("-"*30)
    return df
train=FE(train)
test=FE(test)

print("group age&general feature")
float_cols=['clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A/tbp_lv_Aext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C-tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H-tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/cos_tbp_lv_symm_2axis_angle', 'V_tbp_lv', 'S_tbp_lv']
for col in tqdm(float_cols):
    for c in [['age_approx','anatom_site_general']]:
        if len(c)>1:#c=[col1,……,coln]
            colname='_'.join(c)
        else:
            colname=c[0]
        tmp=train.groupby(c)[col].mean().reset_index().rename(columns={col:f"mean_{colname}_{col}"})
        train[f"mean_{colname}_{col}"] = train.groupby(c)[col].transform(np.mean)
        test=test.merge(tmp,on=c,how='left')

        tmp=train.groupby(c)[col].min().reset_index().rename(columns={col:f"min_{colname}_{col}"})
        train[f"min_{colname}_{col}"] = train.groupby(c)[col].transform(np.min)
        test=test.merge(tmp,on=c,how='left')

        tmp=train.groupby(c)[col].max().reset_index().rename(columns={col:f"max_{colname}_{col}"})
        train[f"max_{colname}_{col}"] = train.groupby(c)[col].transform(np.max)
        test=test.merge(tmp,on=c,how='left')

        tmp=train.groupby(c)[col].std().reset_index().rename(columns={col:f"std_{colname}_{col}"})
        train[f"std_{colname}_{col}"] = train.groupby(c)[col].transform(np.std)
        test=test.merge(tmp,on=c,how='left')

        tmp=train.groupby(c)[col].skew().reset_index().rename(columns={col:f"skew_{colname}_{col}"})
        train[f"skew_{colname}_{col}"] = train.groupby(c)[col].transform(scipy.stats.skew)
        test=test.merge(tmp,on=c,how='left')

        tmp=train.groupby(c)[col].median().reset_index().rename(columns={col:f"median_{colname}_{col}"})
        train[f"median_{colname}_{col}"] = train.groupby(c)[col].transform(np.median)
        test=test.merge(tmp,on=c,how='left') 
        
print("shift feature")
train=train.sort_values(['patient_id','age_approx'])
test=test.sort_values(['patient_id','age_approx'])
for gap in [1,2]:
    for col in tqdm(float_cols):
        train[f"{col}_diff_{gap}"]=train.groupby(['patient_id'])[col].diff(gap)
        train[f"{col}_diff_{gap}"]=train[f"{col}_diff_{gap}"].fillna(0)
        test[f"{col}_diff_{gap}"]=test.groupby(['patient_id'])[col].diff(gap)
        test[f"{col}_diff_{gap}"]=test[f"{col}_diff_{gap}"].fillna(0)
        
        train[f"{col}_groupanatom_site_general_diff_{gap}"]=train.groupby(['patient_id','anatom_site_general'])[col].diff(gap)
        train[f"{col}_groupanatom_site_general_diff_{gap}"]=train[f"{col}_groupanatom_site_general_diff_{gap}"].fillna(0)
        test[f"{col}_groupanatom_site_general_diff_{gap}"]=test.groupby(['patient_id','anatom_site_general'])[col].diff(gap)
        test[f"{col}_groupanatom_site_general_diff_{gap}"]=test[f"{col}_groupanatom_site_general_diff_{gap}"].fillna(0)

#考虑到内存问题先试试前50个
for col in float_cols[:50]:
    train[f"{col}_diff_2-diff_1"]=train[f"{col}_diff_2"]-train[f"{col}_diff_1"]
    test[f"{col}_diff_2-diff_1"]=test[f"{col}_diff_2"]-test[f"{col}_diff_1"]

train.replace([np.inf, -np.inf], np.nan, inplace=True)    
test.replace([np.inf, -np.inf], np.nan, inplace=True)

for col in test.drop(['isic_id','patient_id','combined_anatomical_site'],axis=1).columns:
    skew=train[col].skew()
    if abs(skew)>0.5:
        min_value=train[col].min()
        train[col]=train[col]-min_value
        test[col]=test[col]-min_value
        
        #print(f"col:{col},skew:{skew}")
        train[col]=np.log1p(train[col])
        test[col]=np.log1p(test[col])

train.head()

Although this dataset is not large and theoretically does not exceed memory capacity, this function was still used.

In [ ]:
#遍历表格df的所有列修改数据类型减少内存使用
def reduce_mem_usage(df, float16_as32=True):
    #memory_usage()是df每列的内存使用量,sum是对它们求和, B->KB->MB
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:#遍历每列的列名
        col_type = df[col].dtype#列名的type
        if col_type != object and str(col_type)!='category':#不是object也就是说这里处理的是数值类型的变量
            c_min,c_max = df[col].min(),df[col].max() #求出这列的最大值和最小值
            if str(col_type)[:3] == 'int':#如果是int类型的变量,不管是int8,int16,int32还是int64
                #如果这列的取值范围是在int8的取值范围内,那就对类型进行转换 (-128 到 127)
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                #如果这列的取值范围是在int16的取值范围内,那就对类型进行转换(-32,768 到 32,767)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                #如果这列的取值范围是在int32的取值范围内,那就对类型进行转换(-2,147,483,648到2,147,483,647)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                #如果这列的取值范围是在int64的取值范围内,那就对类型进行转换(-9,223,372,036,854,775,808到9,223,372,036,854,775,807)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:#如果是浮点数类型.
                #如果数值在float16的取值范围内,如果觉得需要更高精度可以考虑float32
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    if float16_as32:#如果数据需要更高的精度可以选择float32
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float16)  
                #如果数值在float32的取值范围内，对它进行类型转换
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                #如果数值在float64的取值范围内，对它进行类型转换
                else:
                    df[col] = df[col].astype(np.float64)
    #计算一下结束后的内存
    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    #相比一开始的内存减少了百分之多少
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df
train=reduce_mem_usage(train, float16_as32=True)
test=reduce_mem_usage(test, float16_as32=True)

## 5.Metric:<a href="https://www.kaggle.com/code/yunsuxiaozi/isic-2024-metric-pauc">pauc</a>

In [ ]:
#这里使用官方的评估指标pauc
def pauc_above_tpr(y_true, y_pred):
    min_tpr=0.8
    v_gt = abs(np.asarray(y_true)-1)
    v_pred = np.array([1.0 - x for x in y_pred])
    max_fpr = abs(1-min_tpr)
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return 'pauc',partial_auc,True

## 6.Model training

- We use 'GroupKFold' here to prevent data leakage between patients.Due to the involvement of different patients in the training and testing sets, the label distribution varies among different patients, therefore 'StratifiedGroupKFold' is not used.

- Due to the small proportion of positive samples, 'early_stop' is not used here.

- If the data is not undersampled, the sample will be imbalanced; If undersampling the data leads to a small total sample size and poor performance of the trained model, we will sample 100000 negative samples and weight the positive samples.

In [ ]:
choose_cols=[col for col in test.columns if train[col].dtype!=object]

#相关性太高而drop
#drop_cols=[]
# metric=train[choose_cols].corr().values
# for i in range(len(metric)):
#     for j in range(i+1,len(metric)):
#         if abs(metric[i][j])>0.95:
#             drop_cols+=[choose_cols[j]]
#     print(f"i:{i}")
#print(f"drop_cols={drop_cols}")

#和其他列相关性达到0.95,只保留一个即可
drop_cols=['mean_sex_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'tbp_lv_perimeterMM', 'clin_size_long_diam_mm-mean_patient_id_clin_size_long_diam_mm/std_patient_id_clin_size_long_diam_mm', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_minorAxisMM', 'tbp_lv_areaMM2-mean_patient_id_tbp_lv_areaMM2/std_patient_id_tbp_lv_areaMM2', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'lesion_shape_index', 'color_shape_composite_index', 'tbp_lv_norm_color', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_L-tbp_lv_Lext', 'luminance_contrast', 'lesion_color_difference', 'std_dev_contrast', 'tbp_lv_eccentricity-mean_patient_id_tbp_lv_eccentricity/std_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'border_complexity', 'symmetry_border_consistency', 'shape_complexity_index', 'tbp_lv_radial_color_std_max', 'tbp_lv_perimeterMM-mean_patient_id_tbp_lv_perimeterMM/std_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'symmetry_border_consistency', 'tbp_lv_x-mean_patient_id_tbp_lv_x/std_patient_id_tbp_lv_x', '3d_position_distance', 'min_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'color_shape_composite_index', 'lesion_color_difference', 'std_dev_contrast', 'std_dev_contrast', 'symmetry_border_consistency', 'shape_complexity_index', 'shape_complexity_index', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'comprehensive_lesion_index', 'median_age_approx_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_upper extremity', 'median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'sum_patient_id_tbp_tile_type', 'tbp_lv_location_Unknown', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_attribution_tbp_lv_dnn_lesion_confidence', 'attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'std_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'tbp_lv_location_simple_Torso Back', 'max_age_approx_anatom_site_general_tbp_lv_y', 'tbp_lv_location_simple_Torso Front', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L*tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle-mean_patient_id_sin_tbp_lv_symm_2axis_angle/std_patient_id_sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle-mean_patient_id_cos_tbp_lv_symm_2axis_angle/std_patient_id_cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'sum_patient_id_copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY-NC', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_attribution_Memorial Sloan Kettering Cancer Center', 'sum_patient_id_attribution_Department of Dermatology, Hospital Clínic de Barcelona', 'sum_patient_id_attribution_University Hospital of Basel', 'sum_patient_id_attribution_Frazer Institute, The University of Queensland, Dermatology Research Centre', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_attribution_Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'median_patient_id_clin_size_long_diam_mm', 'mean_patient_id_tbp_lv_perimeterMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_areaMM2', 'median_patient_id_tbp_lv_perimeterMM', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_A', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A/mean_patient_id_tbp_lv_A', 'tbp_lv_A+tbp_lv_Aext-mean_patient_id_tbp_lv_A+tbp_lv_Aext/std_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_Aext/mean_patient_id_tbp_lv_Aext', 'median_patient_id_tbp_lv_B', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/mean_patient_id_tbp_lv_B', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_Bext/mean_patient_id_tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_C', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/mean_patient_id_tbp_lv_C', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_Cext/mean_patient_id_tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'ptp_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_H', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/mean_patient_id_tbp_lv_H', 'median_patient_id_tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_Hext/mean_patient_id_tbp_lv_Hext', 'median_patient_id_tbp_lv_L', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L/mean_patient_id_tbp_lv_L', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'skew_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_Lext/mean_patient_id_tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_areaMM2', 'mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_area_perim_ratio', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_area_perim_ratio', 'std_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'tbp_lv_area_perim_ratio/mean_patient_id_tbp_lv_area_perim_ratio', 'median_patient_id_tbp_lv_color_std_mean', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'min_patient_id_tbp_lv_norm_color', 'ptp_patient_id_tbp_lv_color_std_mean', 'std_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_color_std_mean/mean_patient_id_tbp_lv_color_std_mean', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_deltaA', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'min_patient_id_tbp_lv_A-tbp_lv_Aext', 'max_patient_id_tbp_lv_A-tbp_lv_Aext', 'std_patient_id_tbp_lv_A-tbp_lv_Aext', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'skew_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext-mean_patient_id_tbp_lv_A-tbp_lv_Aext/std_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext/mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_deltaB', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'min_patient_id_tbp_lv_B-tbp_lv_Bext', 'max_patient_id_tbp_lv_B-tbp_lv_Bext', 'std_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'skew_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext-mean_patient_id_tbp_lv_B-tbp_lv_Bext/std_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext/mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_deltaL', 'mean_patient_id_tbp_lv_deltaLB', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaL', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_deltaLB', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_deltaLB', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaL/mean_patient_id_tbp_lv_deltaL', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLBnorm', 'ptp_patient_id_tbp_lv_deltaLBnorm', 'std_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_deltaLBnorm/mean_patient_id_tbp_lv_deltaLBnorm', 'median_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'skew_patient_id_tbp_lv_nevi_confidence', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_norm_border', 'std_patient_id_tbp_lv_symm_2axis', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_norm_border/mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_stdL', 'ptp_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_stdL/mean_patient_id_tbp_lv_stdL', 'median_patient_id_tbp_lv_stdLExt', 'ptp_patient_id_tbp_lv_stdLExt', 'tbp_lv_stdLExt/mean_patient_id_tbp_lv_stdLExt', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_z', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'min_patient_id_tbp_lv_A*tbp_lv_Aext', 'max_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A/tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A/tbp_lv_Aext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'skew_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'min_patient_id_tbp_lv_C*tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'skew_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C-tbp_lv_Cext', 'mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext-mean_patient_id_tbp_lv_C/tbp_lv_Cext/std_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext/mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'min_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'skew_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H+tbp_lv_Hext/mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H-tbp_lv_Hext', 'mean_patient_id_tbp_lv_H/tbp_lv_Hext', 'ptp_patient_id_tbp_lv_H-tbp_lv_Hext', 'max_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H/tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext-mean_patient_id_tbp_lv_H/tbp_lv_Hext/std_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L/tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext/mean_patient_id_tbp_lv_L/tbp_lv_Lext', 'median_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_patient_id_cos_tbp_lv_symm_2axis_angle', 'median_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_V_tbp_lv', 'V_tbp_lv/mean_patient_id_V_tbp_lv', 'ptp_patient_id_S_tbp_lv', 'S_tbp_lv/mean_patient_id_S_tbp_lv', 'sum_patient_id_tbp_lv_location_simple', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_tbp_lv_location_simple', 'skew_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_head/neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_tbp_lv_location_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'tbp_lv_B_count', 'sum_patient_id_tbp_lv_location_Torso Back Top Third', 'sum_patient_id_tbp_lv_location_Torso Back Middle Third', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'skew_patient_id_tbp_lv_location_simple_Torso Back', 'std_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_Torso Front Top Half', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'skew_patient_id_tbp_lv_location_simple_Torso Front', 'std_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Left Arm', 'sum_patient_id_tbp_lv_location_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_Left Leg', 'std_patient_id_tbp_lv_location_Right Leg', 'std_patient_id_tbp_lv_location_Left Arm', 'skew_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Torso Front', 'std_patient_id_tbp_lv_location_Torso Back', 'median_age_approx_anatom_site_general_clin_size_long_diam_mm', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'std_age_approx_anatom_site_general_tbp_lv_areaMM2', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'median_age_approx_anatom_site_general_tbp_lv_A', 'median_age_approx_anatom_site_general_tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_areaMM2', 'mean_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'std_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'skew_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_color_std_mean', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'min_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_deltaA', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'min_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_deltaB', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'min_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_deltaL', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_deltaLB', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLB', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_deltaLB', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'std_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_sin_tbp_lv_symm_2axis_angle', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_stdLExt', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_z', 'median_age_approx_anatom_site_general_tbp_lv_A+tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'min_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'skew_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C-tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'max_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'std_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'max_age_approx_anatom_site_general_S_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_S_tbp_lv', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_norm_color_diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_1', 'tbp_lv_A-tbp_lv_Aext_diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B-tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_deltaLB_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L/tbp_lv_Lext_diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_radial_color_std_max_diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_1', 'tbp_lv_A*tbp_lv_Aext_diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C/tbp_lv_Cext_diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_H*tbp_lv_Hext_diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_H/tbp_lv_Hext_diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_norm_color_diff_2', 'tbp_lv_norm_color_groupanatom_site_general_diff_2', 'tbp_lv_A-tbp_lv_Aext_diff_2', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B-tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_deltaLB_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L/tbp_lv_Lext_diff_2', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_radial_color_std_max_diff_2', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2', 'tbp_lv_A*tbp_lv_Aext_diff_2', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C/tbp_lv_Cext_diff_2', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_H*tbp_lv_Hext_diff_2', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_H/tbp_lv_Hext_diff_2', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2']

choose_cols=[col for col in choose_cols if col not in drop_cols]
print(f"len(choose_cols):{len(choose_cols)}")

cols_name={}
for i in range(len(choose_cols)):
    cols_name[choose_cols[i]]=f"cols_{i}"

def fit_and_predict(train_feats=train,test_feats=test,model=None,num_folds=10,name='lgb'):
    X=train_feats[choose_cols].copy().rename(columns=cols_name)
    y=train_feats['target'].copy()
    patient_id=train_feats['patient_id'].copy()
    oof_pred=np.zeros((len(X)))
    test_X=test_feats[choose_cols].copy().rename(columns=cols_name)
    test_pred_pro=np.zeros((num_folds,len(test_X)))
     
    #k折交叉验证
    gkf = GroupKFold(n_splits=num_folds)#,shuffle=True
    for fold, (train_index, valid_index) in (enumerate(gkf.split(X,y,patient_id))):
        print(f"name {name},fold:{fold}")

        X_train, X_valid = X.iloc[train_index].reset_index(drop=True), X.iloc[valid_index].reset_index(drop=True)
        y_train, y_valid = y.iloc[train_index].reset_index(drop=True), y.iloc[valid_index].reset_index(drop=True)
        
        if 'lgb' in name:
            #对训练数据label=0的数据做采样1W
            zero_index=np.where(y_train==0)[0]
            one_index=np.where(y_train==1)[0]
            np.random.shuffle(zero_index)
            np.random.shuffle(one_index)
            total_index=list(zero_index[:10000])+list(one_index)+list(one_index)[:10]
            X_train=X_train.iloc[total_index]
            y_train=y_train.iloc[total_index]
            model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
                          callbacks=[log_evaluation(100)],
                          eval_metric=pauc_above_tpr
                         )
        if 'xgb' in name:
            #对训练数据label=0的数据做采样1W
            zero_index=np.where(y_train==0)[0]
            one_index=np.where(y_train==1)[0]
            np.random.shuffle(zero_index)
            np.random.shuffle(one_index)
            total_index=list(zero_index[:20000])+list(one_index)+list(one_index)[:50]
            X_train=X_train.iloc[total_index]
            y_train=y_train.iloc[total_index]
            model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
                          verbose=100,
                         )
        if 'cat' in name:
            #对训练数据label=0的数据做采样1W
            zero_index=np.where(y_train==0)[0]
            one_index=np.where(y_train==1)[0]
            np.random.shuffle(zero_index)
            np.random.shuffle(one_index)
            total_index=list(zero_index[:8000])+list(one_index)+list(one_index)[:5]
            X_train=X_train.iloc[total_index]
            y_train=y_train.iloc[total_index]
            model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
                          verbose=100
                         )
        
        oof_pred[valid_index]=model.predict_proba(X_valid)[:,1]
        test_pred_pro[fold]=model.predict_proba(test_X)[:,1]
        
    min_value,max_value=np.min(oof_pred),np.max(oof_pred)
    
    oof_pred=(oof_pred-min_value)/(max_value-min_value)

    test_pred_pro=test_pred_pro.mean(axis=0)
    
    print(f"name:{name},pauc:{pauc_above_tpr(y.values.astype(np.int8),oof_pred)}")
    
    return oof_pred,(test_pred_pro-min_value)/(max_value-min_value)

#### optuna find best lgb_params

In [ ]:
#这是最新的optuna调参代码,应该不会再更新了,如果要使用代码找参数需要确认chooose_cols和程序里的choose_cols是否对应上
# import optuna#自动超参数优化软件框架

# choose_cols=[col for col in test.columns if train[col].dtype!=object]

# #相关性太高而drop
# #drop_cols=[]
# # metric=train[choose_cols].corr().values
# # for i in range(len(metric)):
# #     for j in range(i+1,len(metric)):
# #         if abs(metric[i][j])>0.95:
# #             drop_cols+=[choose_cols[j]]
# #     print(f"i:{i}")
# #print(f"drop_cols={drop_cols}")

# #和其他列相关性达到0.95,只保留一个即可 0829晚上找的
# drop_cols=['mean_sex_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'tbp_lv_perimeterMM', 'clin_size_long_diam_mm-mean_patient_id_clin_size_long_diam_mm/std_patient_id_clin_size_long_diam_mm', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_minorAxisMM', 'tbp_lv_areaMM2-mean_patient_id_tbp_lv_areaMM2/std_patient_id_tbp_lv_areaMM2', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'lesion_shape_index', 'color_shape_composite_index', 'tbp_lv_norm_color', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_L-tbp_lv_Lext', 'luminance_contrast', 'lesion_color_difference', 'std_dev_contrast', 'tbp_lv_eccentricity-mean_patient_id_tbp_lv_eccentricity/std_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'border_complexity', 'symmetry_border_consistency', 'shape_complexity_index', 'tbp_lv_radial_color_std_max', 'tbp_lv_perimeterMM-mean_patient_id_tbp_lv_perimeterMM/std_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'symmetry_border_consistency', 'tbp_lv_x-mean_patient_id_tbp_lv_x/std_patient_id_tbp_lv_x', '3d_position_distance', 'min_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'color_shape_composite_index', 'lesion_color_difference', 'std_dev_contrast', 'std_dev_contrast', 'symmetry_border_consistency', 'shape_complexity_index', 'shape_complexity_index', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'comprehensive_lesion_index', 'median_age_approx_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_upper extremity', 'median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'sum_patient_id_tbp_tile_type', 'tbp_lv_location_Unknown', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_attribution_tbp_lv_dnn_lesion_confidence', 'attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'std_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'tbp_lv_location_simple_Torso Back', 'max_age_approx_anatom_site_general_tbp_lv_y', 'tbp_lv_location_simple_Torso Front', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L*tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle-mean_patient_id_sin_tbp_lv_symm_2axis_angle/std_patient_id_sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle-mean_patient_id_cos_tbp_lv_symm_2axis_angle/std_patient_id_cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'sum_patient_id_copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY-NC', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_attribution_Memorial Sloan Kettering Cancer Center', 'sum_patient_id_attribution_Department of Dermatology, Hospital Clínic de Barcelona', 'sum_patient_id_attribution_University Hospital of Basel', 'sum_patient_id_attribution_Frazer Institute, The University of Queensland, Dermatology Research Centre', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_attribution_Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'median_patient_id_clin_size_long_diam_mm', 'mean_patient_id_tbp_lv_perimeterMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_areaMM2', 'median_patient_id_tbp_lv_perimeterMM', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_A', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A/mean_patient_id_tbp_lv_A', 'tbp_lv_A+tbp_lv_Aext-mean_patient_id_tbp_lv_A+tbp_lv_Aext/std_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_Aext/mean_patient_id_tbp_lv_Aext', 'median_patient_id_tbp_lv_B', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/mean_patient_id_tbp_lv_B', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_Bext/mean_patient_id_tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_C', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/mean_patient_id_tbp_lv_C', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_Cext/mean_patient_id_tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'ptp_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_H', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/mean_patient_id_tbp_lv_H', 'median_patient_id_tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_Hext/mean_patient_id_tbp_lv_Hext', 'median_patient_id_tbp_lv_L', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L/mean_patient_id_tbp_lv_L', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'skew_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_Lext/mean_patient_id_tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_areaMM2', 'mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_area_perim_ratio', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_area_perim_ratio', 'std_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'tbp_lv_area_perim_ratio/mean_patient_id_tbp_lv_area_perim_ratio', 'median_patient_id_tbp_lv_color_std_mean', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'min_patient_id_tbp_lv_norm_color', 'ptp_patient_id_tbp_lv_color_std_mean', 'std_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_color_std_mean/mean_patient_id_tbp_lv_color_std_mean', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_deltaA', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'min_patient_id_tbp_lv_A-tbp_lv_Aext', 'max_patient_id_tbp_lv_A-tbp_lv_Aext', 'std_patient_id_tbp_lv_A-tbp_lv_Aext', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'skew_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext-mean_patient_id_tbp_lv_A-tbp_lv_Aext/std_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext/mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_deltaB', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'min_patient_id_tbp_lv_B-tbp_lv_Bext', 'max_patient_id_tbp_lv_B-tbp_lv_Bext', 'std_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'skew_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext-mean_patient_id_tbp_lv_B-tbp_lv_Bext/std_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext/mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_deltaL', 'mean_patient_id_tbp_lv_deltaLB', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaL', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_deltaLB', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_deltaLB', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaL/mean_patient_id_tbp_lv_deltaL', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLBnorm', 'ptp_patient_id_tbp_lv_deltaLBnorm', 'std_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_deltaLBnorm/mean_patient_id_tbp_lv_deltaLBnorm', 'median_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'skew_patient_id_tbp_lv_nevi_confidence', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_norm_border', 'std_patient_id_tbp_lv_symm_2axis', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_norm_border/mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_stdL', 'ptp_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_stdL/mean_patient_id_tbp_lv_stdL', 'median_patient_id_tbp_lv_stdLExt', 'ptp_patient_id_tbp_lv_stdLExt', 'tbp_lv_stdLExt/mean_patient_id_tbp_lv_stdLExt', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_z', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'min_patient_id_tbp_lv_A*tbp_lv_Aext', 'max_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A/tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A/tbp_lv_Aext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'skew_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'min_patient_id_tbp_lv_C*tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'skew_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C-tbp_lv_Cext', 'mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext-mean_patient_id_tbp_lv_C/tbp_lv_Cext/std_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext/mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'min_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'skew_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H+tbp_lv_Hext/mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H-tbp_lv_Hext', 'mean_patient_id_tbp_lv_H/tbp_lv_Hext', 'ptp_patient_id_tbp_lv_H-tbp_lv_Hext', 'max_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H/tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext-mean_patient_id_tbp_lv_H/tbp_lv_Hext/std_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L/tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext/mean_patient_id_tbp_lv_L/tbp_lv_Lext', 'median_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_patient_id_cos_tbp_lv_symm_2axis_angle', 'median_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_V_tbp_lv', 'V_tbp_lv/mean_patient_id_V_tbp_lv', 'ptp_patient_id_S_tbp_lv', 'S_tbp_lv/mean_patient_id_S_tbp_lv', 'sum_patient_id_tbp_lv_location_simple', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_tbp_lv_location_simple', 'skew_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_head/neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_tbp_lv_location_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'tbp_lv_B_count', 'sum_patient_id_tbp_lv_location_Torso Back Top Third', 'sum_patient_id_tbp_lv_location_Torso Back Middle Third', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'skew_patient_id_tbp_lv_location_simple_Torso Back', 'std_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_Torso Front Top Half', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'skew_patient_id_tbp_lv_location_simple_Torso Front', 'std_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Left Arm', 'sum_patient_id_tbp_lv_location_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_Left Leg', 'std_patient_id_tbp_lv_location_Right Leg', 'std_patient_id_tbp_lv_location_Left Arm', 'skew_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Torso Front', 'std_patient_id_tbp_lv_location_Torso Back', 'median_age_approx_anatom_site_general_clin_size_long_diam_mm', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'std_age_approx_anatom_site_general_tbp_lv_areaMM2', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'median_age_approx_anatom_site_general_tbp_lv_A', 'median_age_approx_anatom_site_general_tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_areaMM2', 'mean_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'std_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'skew_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_color_std_mean', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'min_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_deltaA', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'min_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_deltaB', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'min_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_deltaL', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_deltaLB', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLB', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_deltaLB', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'std_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_sin_tbp_lv_symm_2axis_angle', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_stdLExt', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_z', 'median_age_approx_anatom_site_general_tbp_lv_A+tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'min_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'skew_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C-tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'max_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'std_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'max_age_approx_anatom_site_general_S_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_S_tbp_lv', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_norm_color_diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_1', 'tbp_lv_A-tbp_lv_Aext_diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B-tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_deltaLB_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L/tbp_lv_Lext_diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_radial_color_std_max_diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_1', 'tbp_lv_A*tbp_lv_Aext_diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C/tbp_lv_Cext_diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_H*tbp_lv_Hext_diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_H/tbp_lv_Hext_diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_norm_color_diff_2', 'tbp_lv_norm_color_groupanatom_site_general_diff_2', 'tbp_lv_A-tbp_lv_Aext_diff_2', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B-tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_deltaLB_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L/tbp_lv_Lext_diff_2', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_radial_color_std_max_diff_2', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2', 'tbp_lv_A*tbp_lv_Aext_diff_2', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C/tbp_lv_Cext_diff_2', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_H*tbp_lv_Hext_diff_2', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_H/tbp_lv_Hext_diff_2', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_norm_color_diff_2-diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A-tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B-tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_deltaLB_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L/tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_radial_color_std_max_diff_2-diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A*tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C/tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H*tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H/tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1']

# choose_cols=[col for col in choose_cols if col not in drop_cols]
# print(f"len(choose_cols):{len(choose_cols)}")

# cols_name={}
# for i in range(len(choose_cols)):
#     cols_name[choose_cols[i]]=f"cols_{i}"

# def objective(trial):
#     param = {
#         "boosting_type": "gbdt",
#         "objective": "binary",
#         "metric": "auc",
#         'random_state': trial.suggest_int('random_state',2024,2024),
#         'n_estimators': trial.suggest_int('n_estimators', 100,300),
#         'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
#         'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),#对数分布的建议值
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),#浮点数
#         'subsample': trial.suggest_float('subsample', 0.5, 1),
#         'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.5, log=True),
#         'num_leaves' : trial.suggest_int('num_leaves', 8, 64),#整数
#         'min_child_samples': trial.suggest_int('min_child_samples', 2, 100),
#         'scale_pos_weight': 2.5,
#         "verbose": -1,
#        # 'device':'gpu','gpu_use_dp':True,#这行GPU环境的参数,想在CPU环境下运行注释这行代码
#     }
#     model = LGBMClassifier(**param)  
    
#     X=train[choose_cols].copy().rename(columns=cols_name)
#     y=train['target'].copy()
#     patient_id=train['patient_id'].copy()
#     oof_pred=np.zeros((len(X)))
     
#     num_folds=10
#     #k折交叉验证
#     gkf = GroupKFold(n_splits=num_folds)#,shuffle=True
#     for fold, (train_index, valid_index) in (enumerate(gkf.split(X,y,patient_id))):
        
#         print(f"fold:{fold}")
        
#         X_train, X_valid = X.iloc[train_index].reset_index(drop=True), X.iloc[valid_index].reset_index(drop=True)
#         y_train, y_valid = y.iloc[train_index].reset_index(drop=True), y.iloc[valid_index].reset_index(drop=True)
        
#         #对训练数据label=0的数据做采样1W
#         zero_index=np.where(y_train==0)[0]
#         one_index=np.where(y_train==1)[0]
#         np.random.shuffle(zero_index)
#         np.random.shuffle(one_index)
#         total_index=list(zero_index[:10000])+list(one_index)+list(one_index)[:10]
#         X_train=X_train.iloc[total_index]
#         y_train=y_train.iloc[total_index]
        
#         model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
#                           callbacks=[log_evaluation(100)],
#                          )
#         oof_pred[valid_index]=model.predict_proba(X_valid)[:,1]
#     pauc=pauc_above_tpr(y.values.astype(np.int8),oof_pred)[1]
    
#     return pauc
# #创建的研究命名,找最大值.
# study = optuna.create_study(direction='maximize', study_name='Optimize boosting hyperparameters')
# #目标函数,尝试的次数  
# study.optimize(objective, n_trials=80)
# lgbm_params=study.best_trial.params
# #输出最佳的参数
# print('lgbm_params=', lgbm_params)

#### optuna find best xgb_params

In [ ]:
#这是最新的找参数代码,应该不会再更新了,如果要使用代码找参数需要确认chooose_cols和程序里的choose_cols是否对应上
# import optuna#自动超参数优化软件框架

# choose_cols=[col for col in test.columns if train[col].dtype!=object]

# #相关性太高而drop
# #drop_cols=[]
# # metric=train[choose_cols].corr().values
# # for i in range(len(metric)):
# #     for j in range(i+1,len(metric)):
# #         if abs(metric[i][j])>0.95:
# #             drop_cols+=[choose_cols[j]]
# #     print(f"i:{i}")
# #print(f"drop_cols={drop_cols}")

# #和其他列相关性达到0.95,只保留一个即可 0829晚上找的
# drop_cols=['mean_sex_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'tbp_lv_perimeterMM', 'clin_size_long_diam_mm-mean_patient_id_clin_size_long_diam_mm/std_patient_id_clin_size_long_diam_mm', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_minorAxisMM', 'tbp_lv_areaMM2-mean_patient_id_tbp_lv_areaMM2/std_patient_id_tbp_lv_areaMM2', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'lesion_shape_index', 'color_shape_composite_index', 'tbp_lv_norm_color', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_L-tbp_lv_Lext', 'luminance_contrast', 'lesion_color_difference', 'std_dev_contrast', 'tbp_lv_eccentricity-mean_patient_id_tbp_lv_eccentricity/std_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'border_complexity', 'symmetry_border_consistency', 'shape_complexity_index', 'tbp_lv_radial_color_std_max', 'tbp_lv_perimeterMM-mean_patient_id_tbp_lv_perimeterMM/std_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'symmetry_border_consistency', 'tbp_lv_x-mean_patient_id_tbp_lv_x/std_patient_id_tbp_lv_x', '3d_position_distance', 'min_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'color_shape_composite_index', 'lesion_color_difference', 'std_dev_contrast', 'std_dev_contrast', 'symmetry_border_consistency', 'shape_complexity_index', 'shape_complexity_index', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'comprehensive_lesion_index', 'median_age_approx_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_upper extremity', 'median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'sum_patient_id_tbp_tile_type', 'tbp_lv_location_Unknown', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_attribution_tbp_lv_dnn_lesion_confidence', 'attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'std_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'tbp_lv_location_simple_Torso Back', 'max_age_approx_anatom_site_general_tbp_lv_y', 'tbp_lv_location_simple_Torso Front', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L*tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle-mean_patient_id_sin_tbp_lv_symm_2axis_angle/std_patient_id_sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle-mean_patient_id_cos_tbp_lv_symm_2axis_angle/std_patient_id_cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'sum_patient_id_copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY-NC', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_attribution_Memorial Sloan Kettering Cancer Center', 'sum_patient_id_attribution_Department of Dermatology, Hospital Clínic de Barcelona', 'sum_patient_id_attribution_University Hospital of Basel', 'sum_patient_id_attribution_Frazer Institute, The University of Queensland, Dermatology Research Centre', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_attribution_Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'median_patient_id_clin_size_long_diam_mm', 'mean_patient_id_tbp_lv_perimeterMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_areaMM2', 'median_patient_id_tbp_lv_perimeterMM', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_A', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A/mean_patient_id_tbp_lv_A', 'tbp_lv_A+tbp_lv_Aext-mean_patient_id_tbp_lv_A+tbp_lv_Aext/std_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_Aext/mean_patient_id_tbp_lv_Aext', 'median_patient_id_tbp_lv_B', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/mean_patient_id_tbp_lv_B', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_Bext/mean_patient_id_tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_C', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/mean_patient_id_tbp_lv_C', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_Cext/mean_patient_id_tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'ptp_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_H', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/mean_patient_id_tbp_lv_H', 'median_patient_id_tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_Hext/mean_patient_id_tbp_lv_Hext', 'median_patient_id_tbp_lv_L', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L/mean_patient_id_tbp_lv_L', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'skew_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_Lext/mean_patient_id_tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_areaMM2', 'mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_area_perim_ratio', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_area_perim_ratio', 'std_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'tbp_lv_area_perim_ratio/mean_patient_id_tbp_lv_area_perim_ratio', 'median_patient_id_tbp_lv_color_std_mean', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'min_patient_id_tbp_lv_norm_color', 'ptp_patient_id_tbp_lv_color_std_mean', 'std_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_color_std_mean/mean_patient_id_tbp_lv_color_std_mean', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_deltaA', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'min_patient_id_tbp_lv_A-tbp_lv_Aext', 'max_patient_id_tbp_lv_A-tbp_lv_Aext', 'std_patient_id_tbp_lv_A-tbp_lv_Aext', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'skew_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext-mean_patient_id_tbp_lv_A-tbp_lv_Aext/std_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext/mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_deltaB', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'min_patient_id_tbp_lv_B-tbp_lv_Bext', 'max_patient_id_tbp_lv_B-tbp_lv_Bext', 'std_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'skew_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext-mean_patient_id_tbp_lv_B-tbp_lv_Bext/std_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext/mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_deltaL', 'mean_patient_id_tbp_lv_deltaLB', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaL', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_deltaLB', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_deltaLB', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaL/mean_patient_id_tbp_lv_deltaL', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLBnorm', 'ptp_patient_id_tbp_lv_deltaLBnorm', 'std_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_deltaLBnorm/mean_patient_id_tbp_lv_deltaLBnorm', 'median_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'skew_patient_id_tbp_lv_nevi_confidence', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_norm_border', 'std_patient_id_tbp_lv_symm_2axis', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_norm_border/mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_stdL', 'ptp_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_stdL/mean_patient_id_tbp_lv_stdL', 'median_patient_id_tbp_lv_stdLExt', 'ptp_patient_id_tbp_lv_stdLExt', 'tbp_lv_stdLExt/mean_patient_id_tbp_lv_stdLExt', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_z', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'min_patient_id_tbp_lv_A*tbp_lv_Aext', 'max_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A/tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A/tbp_lv_Aext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'skew_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'min_patient_id_tbp_lv_C*tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'skew_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C-tbp_lv_Cext', 'mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext-mean_patient_id_tbp_lv_C/tbp_lv_Cext/std_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext/mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'min_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'skew_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H+tbp_lv_Hext/mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H-tbp_lv_Hext', 'mean_patient_id_tbp_lv_H/tbp_lv_Hext', 'ptp_patient_id_tbp_lv_H-tbp_lv_Hext', 'max_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H/tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext-mean_patient_id_tbp_lv_H/tbp_lv_Hext/std_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L/tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext/mean_patient_id_tbp_lv_L/tbp_lv_Lext', 'median_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_patient_id_cos_tbp_lv_symm_2axis_angle', 'median_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_V_tbp_lv', 'V_tbp_lv/mean_patient_id_V_tbp_lv', 'ptp_patient_id_S_tbp_lv', 'S_tbp_lv/mean_patient_id_S_tbp_lv', 'sum_patient_id_tbp_lv_location_simple', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_tbp_lv_location_simple', 'skew_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_head/neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_tbp_lv_location_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'tbp_lv_B_count', 'sum_patient_id_tbp_lv_location_Torso Back Top Third', 'sum_patient_id_tbp_lv_location_Torso Back Middle Third', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'skew_patient_id_tbp_lv_location_simple_Torso Back', 'std_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_Torso Front Top Half', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'skew_patient_id_tbp_lv_location_simple_Torso Front', 'std_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Left Arm', 'sum_patient_id_tbp_lv_location_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_Left Leg', 'std_patient_id_tbp_lv_location_Right Leg', 'std_patient_id_tbp_lv_location_Left Arm', 'skew_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Torso Front', 'std_patient_id_tbp_lv_location_Torso Back', 'median_age_approx_anatom_site_general_clin_size_long_diam_mm', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'std_age_approx_anatom_site_general_tbp_lv_areaMM2', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'median_age_approx_anatom_site_general_tbp_lv_A', 'median_age_approx_anatom_site_general_tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_areaMM2', 'mean_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'std_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'skew_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_color_std_mean', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'min_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_deltaA', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'min_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_deltaB', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'min_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_deltaL', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_deltaLB', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLB', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_deltaLB', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'std_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_sin_tbp_lv_symm_2axis_angle', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_stdLExt', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_z', 'median_age_approx_anatom_site_general_tbp_lv_A+tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'min_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'skew_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C-tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'max_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'std_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'max_age_approx_anatom_site_general_S_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_S_tbp_lv', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_norm_color_diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_1', 'tbp_lv_A-tbp_lv_Aext_diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B-tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_deltaLB_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L/tbp_lv_Lext_diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_radial_color_std_max_diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_1', 'tbp_lv_A*tbp_lv_Aext_diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C/tbp_lv_Cext_diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_H*tbp_lv_Hext_diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_H/tbp_lv_Hext_diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_norm_color_diff_2', 'tbp_lv_norm_color_groupanatom_site_general_diff_2', 'tbp_lv_A-tbp_lv_Aext_diff_2', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B-tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_deltaLB_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L/tbp_lv_Lext_diff_2', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_radial_color_std_max_diff_2', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2', 'tbp_lv_A*tbp_lv_Aext_diff_2', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C/tbp_lv_Cext_diff_2', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_H*tbp_lv_Hext_diff_2', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_H/tbp_lv_Hext_diff_2', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_norm_color_diff_2-diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A-tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B-tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_deltaLB_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L/tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_radial_color_std_max_diff_2-diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A*tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C/tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H*tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H/tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1']

# choose_cols=[col for col in choose_cols if col not in drop_cols]
# print(f"len(choose_cols):{len(choose_cols)}")

# cols_name={}
# for i in range(len(choose_cols)):
#     cols_name[choose_cols[i]]=f"cols_{i}"

# def objective(trial):
#     param = {'objective': 'binary:logistic',
#             'random_state': trial.suggest_int('random_state',Config.seed,Config.seed),
#             'n_estimators': trial.suggest_int('n_estimators', 50, 300),
#             'learning_rate': trial.suggest_float('learning_rate', 1e-4, 0.1, log=True),
#             'max_depth': trial.suggest_int('max_depth', 3, 10),
#             'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
#             'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),#对数分布的建议值
#             'subsample': trial.suggest_float('subsample', 0.5, 1),
#             'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),#浮点数
#             'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
#              'scale_pos_weight': 2.5,#'tree_method':'gpu_hist'
#             }
#     model = XGBClassifier(**param)
    
#     X=train[choose_cols].copy().rename(columns=cols_name)
#     y=train['target'].copy()
#     patient_id=train['patient_id'].copy()
#     oof_pred=np.zeros((len(X)))
     
#     num_folds=10
#     #k折交叉验证
#     gkf = GroupKFold(n_splits=num_folds)#,shuffle=True
#     for fold, (train_index, valid_index) in (enumerate(gkf.split(X,y,patient_id))):
        
#         print(f"fold:{fold}")
        
#         X_train, X_valid = X.iloc[train_index].reset_index(drop=True), X.iloc[valid_index].reset_index(drop=True)
#         y_train, y_valid = y.iloc[train_index].reset_index(drop=True), y.iloc[valid_index].reset_index(drop=True)
        
#         #对训练数据label=0的数据做采样1W
#         zero_index=np.where(y_train==0)[0]
#         one_index=np.where(y_train==1)[0]
#         np.random.shuffle(zero_index)
#         np.random.shuffle(one_index)
#         total_index=list(zero_index[:10000])+list(one_index)+list(one_index)[:10]
#         X_train=X_train.iloc[total_index]
#         y_train=y_train.iloc[total_index]
        
#         model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
#                          verbose=100,
#                          )
#         oof_pred[valid_index]=model.predict_proba(X_valid)[:,1]
#     pauc=pauc_above_tpr(y.values.astype(np.int8),oof_pred)[1]
    
#     return pauc
# #创建的研究命名,找最大值.
# study = optuna.create_study(direction='maximize', study_name='Optimize boosting hyperparameters')
# #目标函数,尝试的次数  
# study.optimize(objective, n_trials=100)
# xgb_params=study.best_trial.params
# #输出最佳的参数
# print('xgb_params=', xgb_params)

#### optuna find best cat_params

In [ ]:
#最新的找参数代码,应该不会再更新了,如果要使用代码找参数需要确认chooose_cols和程序里的choose_cols是否对应上
# import optuna#自动超参数优化软件框架

# choose_cols=[col for col in test.columns if train[col].dtype!=object]

# #相关性太高而drop
# #drop_cols=[]
# # metric=train[choose_cols].corr().values
# # for i in range(len(metric)):
# #     for j in range(i+1,len(metric)):
# #         if abs(metric[i][j])>0.95:
# #             drop_cols+=[choose_cols[j]]
# #     print(f"i:{i}")
# #print(f"drop_cols={drop_cols}")

# #和其他列相关性达到0.95,只保留一个即可 0829晚上找的
# drop_cols=['mean_sex_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'tbp_lv_perimeterMM', 'clin_size_long_diam_mm-mean_patient_id_clin_size_long_diam_mm/std_patient_id_clin_size_long_diam_mm', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext', 'tbp_lv_minorAxisMM', 'tbp_lv_areaMM2-mean_patient_id_tbp_lv_areaMM2/std_patient_id_tbp_lv_areaMM2', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'lesion_shape_index', 'color_shape_composite_index', 'tbp_lv_norm_color', 'tbp_lv_A-tbp_lv_Aext', 'tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_L-tbp_lv_Lext', 'luminance_contrast', 'lesion_color_difference', 'std_dev_contrast', 'tbp_lv_eccentricity-mean_patient_id_tbp_lv_eccentricity/std_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'border_complexity', 'symmetry_border_consistency', 'shape_complexity_index', 'tbp_lv_radial_color_std_max', 'tbp_lv_perimeterMM-mean_patient_id_tbp_lv_perimeterMM/std_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'symmetry_border_consistency', 'tbp_lv_x-mean_patient_id_tbp_lv_x/std_patient_id_tbp_lv_x', '3d_position_distance', 'min_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'color_shape_composite_index', 'lesion_color_difference', 'std_dev_contrast', 'std_dev_contrast', 'symmetry_border_consistency', 'shape_complexity_index', 'shape_complexity_index', 'tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext', 'comprehensive_lesion_index', 'median_age_approx_tbp_lv_dnn_lesion_confidence', 'min_sex_tbp_lv_dnn_lesion_confidence', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'median_sex_tbp_lv_dnn_lesion_confidence', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'std_sex_tbp_lv_dnn_lesion_confidence', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'skew_sex_tbp_lv_dnn_lesion_confidence', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'sex_male', 'sex_female', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_upper extremity', 'median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_tbp_tile_type', 'sum_patient_id_tbp_tile_type', 'tbp_lv_location_Unknown', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'anatom_site_general_head/neck', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'std_attribution_tbp_lv_dnn_lesion_confidence', 'attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'std_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY', 'copyright_license_CC-0', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_sex_male', 'sum_patient_id_sex_female', 'tbp_lv_location_simple_Torso Back', 'max_age_approx_anatom_site_general_tbp_lv_y', 'tbp_lv_location_simple_Torso Front', 'tbp_lv_location_Head & Neck', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_A*tbp_lv_Aext', 'tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext', 'tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext', 'tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext', 'tbp_lv_L*tbp_lv_Lext', 'sin_tbp_lv_symm_2axis_angle-mean_patient_id_sin_tbp_lv_symm_2axis_angle/std_patient_id_sin_tbp_lv_symm_2axis_angle', 'cos_tbp_lv_symm_2axis_angle-mean_patient_id_cos_tbp_lv_symm_2axis_angle/std_patient_id_cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle-mean_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle/std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'sum_patient_id_copyright_license_CC-BY', 'sum_patient_id_copyright_license_CC-BY-NC', 'attribution_ACEMID MIA', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_attribution_Memorial Sloan Kettering Cancer Center', 'sum_patient_id_attribution_Department of Dermatology, Hospital Clínic de Barcelona', 'sum_patient_id_attribution_University Hospital of Basel', 'sum_patient_id_attribution_Frazer Institute, The University of Queensland, Dermatology Research Centre', 'sum_patient_id_copyright_license_CC-0', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_min_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_attribution_Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris', 'tbp_lv_location_simple_Head & Neck', 'tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_tbp_lv_location_Unknown', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'median_patient_id_clin_size_long_diam_mm', 'mean_patient_id_tbp_lv_perimeterMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_areaMM2', 'median_patient_id_tbp_lv_perimeterMM', 'clin_size_long_diam_mm/mean_patient_id_clin_size_long_diam_mm', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_A', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A/mean_patient_id_tbp_lv_A', 'tbp_lv_A+tbp_lv_Aext-mean_patient_id_tbp_lv_A+tbp_lv_Aext/std_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_Aext/mean_patient_id_tbp_lv_Aext', 'median_patient_id_tbp_lv_B', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B/mean_patient_id_tbp_lv_B', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B+tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_Bext/mean_patient_id_tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext-mean_patient_id_tbp_lv_B+tbp_lv_Bext/std_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_C', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C/mean_patient_id_tbp_lv_C', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C+tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C+tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_Cext/mean_patient_id_tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext-mean_patient_id_tbp_lv_C+tbp_lv_Cext/std_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'ptp_patient_id_tbp_lv_C+tbp_lv_Cext', 'median_patient_id_tbp_lv_H', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H/mean_patient_id_tbp_lv_H', 'median_patient_id_tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_Hext/mean_patient_id_tbp_lv_Hext', 'median_patient_id_tbp_lv_L', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L/mean_patient_id_tbp_lv_L', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L+tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'std_patient_id_tbp_lv_L+tbp_lv_Lext', 'mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'skew_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_Lext/mean_patient_id_tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext-mean_patient_id_tbp_lv_L+tbp_lv_Lext/std_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L+tbp_lv_Lext', 'median_patient_id_tbp_lv_areaMM2', 'mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'min_patient_id_tbp_lv_perimeterMM', 'std_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_areaMM2/mean_patient_id_tbp_lv_areaMM2', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM-mean_patient_id_tbp_lv_minorAxisMM/std_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'median_patient_id_tbp_lv_area_perim_ratio', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_area_perim_ratio', 'std_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'tbp_lv_area_perim_ratio/mean_patient_id_tbp_lv_area_perim_ratio', 'median_patient_id_tbp_lv_color_std_mean', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'min_patient_id_tbp_lv_norm_color', 'ptp_patient_id_tbp_lv_color_std_mean', 'std_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_color_std_mean/mean_patient_id_tbp_lv_color_std_mean', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color-mean_patient_id_tbp_lv_norm_color/std_patient_id_tbp_lv_norm_color', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'median_patient_id_tbp_lv_deltaA', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'min_patient_id_tbp_lv_A-tbp_lv_Aext', 'max_patient_id_tbp_lv_A-tbp_lv_Aext', 'std_patient_id_tbp_lv_A-tbp_lv_Aext', 'mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'skew_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext-mean_patient_id_tbp_lv_A-tbp_lv_Aext/std_patient_id_tbp_lv_A-tbp_lv_Aext', 'tbp_lv_A-tbp_lv_Aext/mean_patient_id_tbp_lv_A-tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_deltaB', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'min_patient_id_tbp_lv_B-tbp_lv_Bext', 'max_patient_id_tbp_lv_B-tbp_lv_Bext', 'std_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'skew_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext-mean_patient_id_tbp_lv_B-tbp_lv_Bext/std_patient_id_tbp_lv_B-tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B-tbp_lv_Bext/mean_patient_id_tbp_lv_B-tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B-tbp_lv_Bext', 'median_patient_id_tbp_lv_deltaL', 'mean_patient_id_tbp_lv_deltaLB', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaL', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_deltaLB', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_deltaLB', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaL/mean_patient_id_tbp_lv_deltaL', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB-mean_patient_id_tbp_lv_deltaLB/std_patient_id_tbp_lv_deltaLB', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'max_patient_id_tbp_lv_deltaLB', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLB', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_deltaLB', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'std_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'skew_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_deltaLB/mean_patient_id_tbp_lv_deltaLB', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext-mean_patient_id_tbp_lv_L-tbp_lv_Lext/std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'min_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_deltaLBnorm', 'ptp_patient_id_tbp_lv_deltaLBnorm', 'std_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_deltaLBnorm/mean_patient_id_tbp_lv_deltaLBnorm', 'median_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_eccentricity', 'tbp_lv_eccentricity/mean_patient_id_tbp_lv_eccentricity', 'ptp_patient_id_tbp_lv_minorAxisMM', 'tbp_lv_minorAxisMM/mean_patient_id_tbp_lv_minorAxisMM', 'skew_patient_id_tbp_lv_nevi_confidence', 'median_patient_id_tbp_lv_norm_border', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_norm_border', 'std_patient_id_tbp_lv_symm_2axis', 'mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_norm_border/mean_patient_id_tbp_lv_norm_border', 'median_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_norm_color', 'mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_norm_color/mean_patient_id_tbp_lv_norm_color', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max-mean_patient_id_tbp_lv_radial_color_std_max/std_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_perimeterMM', 'tbp_lv_perimeterMM/mean_patient_id_tbp_lv_perimeterMM', 'median_patient_id_tbp_lv_radial_color_std_max', 'ptp_patient_id_tbp_lv_radial_color_std_max', 'tbp_lv_radial_color_std_max/mean_patient_id_tbp_lv_radial_color_std_max', 'median_patient_id_tbp_lv_stdL', 'ptp_patient_id_tbp_lv_stdL', 'std_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_stdL/mean_patient_id_tbp_lv_stdL', 'median_patient_id_tbp_lv_stdLExt', 'ptp_patient_id_tbp_lv_stdLExt', 'tbp_lv_stdLExt/mean_patient_id_tbp_lv_stdLExt', 'median_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis', 'std_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'tbp_lv_symm_2axis/mean_patient_id_tbp_lv_symm_2axis', 'median_patient_id_tbp_lv_z', 'median_patient_id_tbp_lv_A+tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'min_patient_id_tbp_lv_A*tbp_lv_Aext', 'max_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A+tbp_lv_Aext/mean_patient_id_tbp_lv_A+tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext-mean_patient_id_tbp_lv_A*tbp_lv_Aext/std_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A-tbp_lv_Aext', 'median_patient_id_tbp_lv_A*tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A*tbp_lv_Aext', 'tbp_lv_A*tbp_lv_Aext/mean_patient_id_tbp_lv_A*tbp_lv_Aext', 'median_patient_id_tbp_lv_A/tbp_lv_Aext', 'ptp_patient_id_tbp_lv_A/tbp_lv_Aext', 'median_patient_id_tbp_lv_B+tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'max_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'skew_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B+tbp_lv_Bext/mean_patient_id_tbp_lv_B+tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext-mean_patient_id_tbp_lv_B*tbp_lv_Bext/std_patient_id_tbp_lv_B*tbp_lv_Bext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B-tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext-mean_patient_id_tbp_lv_B/tbp_lv_Bext/std_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_B*tbp_lv_Bext', 'ptp_patient_id_tbp_lv_B*tbp_lv_Bext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_B*tbp_lv_Bext/mean_patient_id_tbp_lv_B*tbp_lv_Bext', 'median_patient_id_tbp_lv_B/tbp_lv_Bext', 'tbp_lv_B/tbp_lv_Bext/mean_patient_id_tbp_lv_B/tbp_lv_Bext', 'median_patient_id_tbp_lv_C+tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'min_patient_id_tbp_lv_C*tbp_lv_Cext', 'max_patient_id_tbp_lv_C*tbp_lv_Cext', 'std_patient_id_tbp_lv_C*tbp_lv_Cext', 'mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'skew_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C+tbp_lv_Cext/mean_patient_id_tbp_lv_C+tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext-mean_patient_id_tbp_lv_C*tbp_lv_Cext/std_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C-tbp_lv_Cext', 'mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext-mean_patient_id_tbp_lv_C/tbp_lv_Cext/std_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_C*tbp_lv_Cext', 'tbp_lv_C*tbp_lv_Cext/mean_patient_id_tbp_lv_C*tbp_lv_Cext', 'median_patient_id_tbp_lv_C/tbp_lv_Cext', 'tbp_lv_C/tbp_lv_Cext/mean_patient_id_tbp_lv_C/tbp_lv_Cext', 'median_patient_id_tbp_lv_H+tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'min_patient_id_tbp_lv_H*tbp_lv_Hext', 'max_patient_id_tbp_lv_H*tbp_lv_Hext', 'mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'skew_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H+tbp_lv_Hext/mean_patient_id_tbp_lv_H+tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext-mean_patient_id_tbp_lv_H*tbp_lv_Hext/std_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_H-tbp_lv_Hext', 'mean_patient_id_tbp_lv_H/tbp_lv_Hext', 'ptp_patient_id_tbp_lv_H-tbp_lv_Hext', 'max_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H/tbp_lv_Hext', 'tbp_lv_H/tbp_lv_Hext-mean_patient_id_tbp_lv_H/tbp_lv_Hext/std_patient_id_tbp_lv_H/tbp_lv_Hext', 'median_patient_id_tbp_lv_H*tbp_lv_Hext', 'tbp_lv_H*tbp_lv_Hext/mean_patient_id_tbp_lv_H*tbp_lv_Hext', 'median_patient_id_tbp_lv_L+tbp_lv_Lext', 'min_patient_id_tbp_lv_L*tbp_lv_Lext', 'max_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L+tbp_lv_Lext/mean_patient_id_tbp_lv_L+tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext-mean_patient_id_tbp_lv_L*tbp_lv_Lext/std_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L-tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L-tbp_lv_Lext', 'tbp_lv_L-tbp_lv_Lext/mean_patient_id_tbp_lv_L-tbp_lv_Lext', 'median_patient_id_tbp_lv_L*tbp_lv_Lext', 'tbp_lv_L*tbp_lv_Lext/mean_patient_id_tbp_lv_L*tbp_lv_Lext', 'median_patient_id_tbp_lv_L/tbp_lv_Lext', 'ptp_patient_id_tbp_lv_L/tbp_lv_Lext', 'tbp_lv_L/tbp_lv_Lext/mean_patient_id_tbp_lv_L/tbp_lv_Lext', 'median_patient_id_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_patient_id_cos_tbp_lv_symm_2axis_angle', 'median_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_tbp_lv_symm_2axis/sin_tbp_lv_symm_2axis_angle', 'ptp_patient_id_V_tbp_lv', 'V_tbp_lv/mean_patient_id_V_tbp_lv', 'ptp_patient_id_S_tbp_lv', 'S_tbp_lv/mean_patient_id_S_tbp_lv', 'sum_patient_id_tbp_lv_location_simple', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_tbp_lv_location_simple', 'skew_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_sex_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'std_patient_id_anatom_site_general_upper extremity', 'sum_patient_id_median_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_anatom_site_general_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_anatom_site_general_head/neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_min_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'std_patient_id_skew_tbp_tile_type_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'skew_patient_id_tbp_lv_location_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'sum_patient_id_median_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'std_patient_id_anatom_site_general_head/neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_mean_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'skew_patient_id_skew_tbp_lv_location_simple_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_attribution_ViDIR Group, Department of Dermatology, Medical University of Vienna', 'sum_patient_id_median_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_attribution_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_mean_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_skew_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_max_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_median_copyright_license_tbp_lv_dnn_lesion_confidence', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'sum_patient_id_std_copyright_license_tbp_lv_dnn_lesion_confidence', 'tbp_lv_B_count', 'tbp_lv_B_count', 'sum_patient_id_tbp_lv_location_Torso Back Top Third', 'sum_patient_id_tbp_lv_location_Torso Back Middle Third', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'skew_patient_id_tbp_lv_location_simple_Torso Back', 'std_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_Torso Front Top Half', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'skew_patient_id_tbp_lv_location_simple_Torso Front', 'std_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Left Arm', 'sum_patient_id_tbp_lv_location_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_attribution_ACEMID MIA', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Torso Front', 'sum_patient_id_tbp_lv_location_simple_Torso Back', 'sum_patient_id_tbp_lv_location_simple_Head & Neck', 'skew_patient_id_tbp_lv_location_simple_Head & Neck', 'std_patient_id_tbp_lv_location_simple_Head & Neck', 'sum_patient_id_tbp_lv_location_simple_Unknown', 'skew_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_simple_Unknown', 'std_patient_id_tbp_lv_location_Left Leg', 'std_patient_id_tbp_lv_location_Right Leg', 'std_patient_id_tbp_lv_location_Left Arm', 'skew_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Right Arm', 'std_patient_id_tbp_lv_location_Torso Front', 'std_patient_id_tbp_lv_location_Torso Back', 'median_age_approx_anatom_site_general_clin_size_long_diam_mm', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'std_age_approx_anatom_site_general_tbp_lv_areaMM2', 'median_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'median_age_approx_anatom_site_general_tbp_lv_A', 'median_age_approx_anatom_site_general_tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_C', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'std_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_areaMM2', 'mean_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'std_age_approx_anatom_site_general_tbp_lv_minorAxisMM', 'skew_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_area_perim_ratio', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_color_std_mean', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'min_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_norm_color', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_norm_color', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_deltaA', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'min_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_deltaB', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'min_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'std_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_deltaL', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_deltaLB', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLB', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_deltaLB', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLB', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLB', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'min_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_deltaLBnorm', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'median_age_approx_anatom_site_general_tbp_lv_eccentricity', 'std_age_approx_anatom_site_general_tbp_lv_perimeterMM', 'skew_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_sin_tbp_lv_symm_2axis_angle', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_nevi_confidence', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_norm_border', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_border', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_norm_color', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'mean_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'median_age_approx_anatom_site_general_tbp_lv_radial_color_std_max', 'std_age_approx_anatom_site_general_tbp_lv_stdL', 'median_age_approx_anatom_site_general_tbp_lv_stdL', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'std_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_stdLExt', 'skew_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_y', 'median_age_approx_anatom_site_general_tbp_lv_z', 'median_age_approx_anatom_site_general_tbp_lv_A+tbp_lv_Aext', 'max_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A-tbp_lv_Aext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'std_age_approx_anatom_site_general_tbp_lv_H-tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A*tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_A/tbp_lv_Aext', 'median_age_approx_anatom_site_general_tbp_lv_B+tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'min_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'max_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'skew_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B-tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'median_age_approx_anatom_site_general_tbp_lv_B*tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C+tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_B/tbp_lv_Bext', 'mean_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'max_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'skew_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C-tbp_lv_Cext', 'mean_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C*tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_C/tbp_lv_Cext', 'median_age_approx_anatom_site_general_tbp_lv_H+tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'min_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'max_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'mean_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'std_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'skew_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H/tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_H*tbp_lv_Hext', 'median_age_approx_anatom_site_general_tbp_lv_L+tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'max_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'skew_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'mean_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L-tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L*tbp_lv_Lext', 'median_age_approx_anatom_site_general_tbp_lv_L/tbp_lv_Lext', 'min_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'max_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'skew_age_approx_anatom_site_general_cos_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*sin_tbp_lv_symm_2axis_angle', 'mean_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'median_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_tbp_lv_symm_2axis*cos_tbp_lv_symm_2axis_angle', 'std_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'max_age_approx_anatom_site_general_S_tbp_lv', 'median_age_approx_anatom_site_general_V_tbp_lv', 'median_age_approx_anatom_site_general_S_tbp_lv', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C+tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L+tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_norm_color_diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_1', 'tbp_lv_A-tbp_lv_Aext_diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B-tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_deltaLB_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_L/tbp_lv_Lext_diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_radial_color_std_max_diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_1', 'tbp_lv_A*tbp_lv_Aext_diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_C/tbp_lv_Cext_diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_1', 'tbp_lv_H*tbp_lv_Hext_diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_H/tbp_lv_Hext_diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_1', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C+tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L+tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_norm_color_diff_2', 'tbp_lv_norm_color_groupanatom_site_general_diff_2', 'tbp_lv_A-tbp_lv_Aext_diff_2', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B-tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_deltaLB_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L-tbp_lv_Lext_diff_2', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_L/tbp_lv_Lext_diff_2', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_radial_color_std_max_diff_2', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2', 'tbp_lv_A*tbp_lv_Aext_diff_2', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2', 'tbp_lv_B*tbp_lv_Bext_diff_2', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_B/tbp_lv_Bext_diff_2', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2', 'tbp_lv_C*tbp_lv_Cext_diff_2', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_C/tbp_lv_Cext_diff_2', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2', 'tbp_lv_H*tbp_lv_Hext_diff_2', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_H/tbp_lv_Hext_diff_2', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2', 'tbp_lv_L*tbp_lv_Lext_diff_2', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B+tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B+tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C+tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C+tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L+tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L+tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_norm_color_diff_2-diff_1', 'tbp_lv_norm_color_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A-tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A-tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B-tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B-tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_deltaLB_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_deltaLB_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L-tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L-tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L/tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L/tbp_lv_Lext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_radial_color_std_max_diff_2-diff_1', 'tbp_lv_radial_color_std_max_groupanatom_site_general_diff_2-diff1', 'tbp_lv_A*tbp_lv_Aext_diff_2-diff_1', 'tbp_lv_A*tbp_lv_Aext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B*tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B*tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_B/tbp_lv_Bext_diff_2-diff_1', 'tbp_lv_B/tbp_lv_Bext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C*tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C*tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_C/tbp_lv_Cext_diff_2-diff_1', 'tbp_lv_C/tbp_lv_Cext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H*tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H*tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_H/tbp_lv_Hext_diff_2-diff_1', 'tbp_lv_H/tbp_lv_Hext_groupanatom_site_general_diff_2-diff1', 'tbp_lv_L*tbp_lv_Lext_diff_2-diff_1', 'tbp_lv_L*tbp_lv_Lext_groupanatom_site_general_diff_2-diff1']

# choose_cols=[col for col in choose_cols if col not in drop_cols]
# print(f"len(choose_cols):{len(choose_cols)}")

# cols_name={}
# for i in range(len(choose_cols)):
#     cols_name[choose_cols[i]]=f"cols_{i}"

# def objective(trial):

#     #https://www.kaggle.com/code/pranjalverma08/catboost-with-optuna-starter-tps-06
#     params = {'iterations':trial.suggest_int("iterations", 50, 300),
#           'od_wait':trial.suggest_int('od_wait', 50, 100),#类似于early stop的过拟合检测器
#           #'task_type':"GPU",
#           'leaf_estimation_method':'Gradient',#'Newton',
#           'bootstrap_type': 'Bernoulli',#每个样本以相同的概率被选中
#           'learning_rate' : trial.suggest_uniform('learning_rate',1e-4,0.25),
#           'reg_lambda': trial.suggest_uniform('reg_lambda',1e-5,100),
#           'subsample': trial.suggest_uniform('subsample',0.5,1),
#           'random_strength': trial.suggest_uniform('random_strength',1,10),
#           'depth': trial.suggest_int('depth',1,15),
#           'min_data_in_leaf': trial.suggest_int('min_data_in_leaf',1,30),
#           'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations',1,15),
#            }
#     model = CatBoostClassifier(**params)
    
#     X=train[choose_cols].copy().rename(columns=cols_name)
#     y=train['target'].copy()
#     patient_id=train['patient_id'].copy()
#     oof_pred=np.zeros((len(X)))
     
#     num_folds=10
#     #k折交叉验证
#     gkf = GroupKFold(n_splits=num_folds)#,shuffle=True
#     for fold, (train_index, valid_index) in (enumerate(gkf.split(X,y,patient_id))):
        
#         print(f"fold:{fold}")
        
#         X_train, X_valid = X.iloc[train_index].reset_index(drop=True), X.iloc[valid_index].reset_index(drop=True)
#         y_train, y_valid = y.iloc[train_index].reset_index(drop=True), y.iloc[valid_index].reset_index(drop=True)
        
#         #对训练数据label=0的数据做采样1W
#         zero_index=np.where(y_train==0)[0]
#         one_index=np.where(y_train==1)[0]
#         np.random.shuffle(zero_index)
#         np.random.shuffle(one_index)
#         total_index=list(zero_index[:10000])+list(one_index)+list(one_index)[:10]
#         X_train=X_train.iloc[total_index]
#         y_train=y_train.iloc[total_index]
        
#         model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],
#                          verbose=100,
#                          )
#         oof_pred[valid_index]=model.predict_proba(X_valid)[:,1]
#     pauc=pauc_above_tpr(y.values.astype(np.int8),oof_pred)[1]
    
#     return pauc
# #创建的研究命名,找最大值.
# study = optuna.create_study(direction='maximize', study_name='Optimize boosting hyperparameters')
# #目标函数,尝试的次数  
# study.optimize(objective, n_trials=50)
# cat_params=study.best_trial.params
# #输出最佳的参数
# print('cat_params=', cat_params)

In [ ]:
#Trial 56 finished with value: 0.1682117632919814 and parameters: 
lgb_params={
        "boosting_type": "gbdt","objective": "binary","metric": "auc",
        'random_state': 2024, 'n_estimators': 275, 
        'reg_alpha': 0.006329813118558037, 'reg_lambda': 0.22366541275310856,
        'colsample_bytree': 0.9045121369263609, 'subsample': 0.6560250299728694,
        'learning_rate': 0.025211401620653728, 'num_leaves': 18, 'min_child_samples': 16,
        'scale_pos_weight': 2.5,"verbose": -1,
        'device':'gpu','gpu_use_dp':True,#这行GPU环境的参数,想在CPU环境下运行注释这行代码
}

lgb_oof_pred_pro,lgb_test_pro=fit_and_predict(model=LGBMClassifier(**lgb_params),num_folds=Config.num_folds,name='lgb')
print(f"lgb_test_pro[:10]:{lgb_test_pro[:10]}")

#Trial 1 finished with value: 0.16312691912494956 and parameters: {
cat_params={'iterations': 1024, 'od_wait': 589,   'task_type':"GPU",
            'leaf_estimation_method':'Newton','bootstrap_type': 'Bernoulli',
            'learning_rate': 0.06565361652314616, 'reg_lambda': 92.76585571631034, 
            'subsample': 0.8310010342463381, 'random_strength': 27.409704119980354, 
            'depth': 7, 'min_data_in_leaf': 24, 'leaf_estimation_iterations': 13}

cat_model=CatBoostClassifier(**cat_params)
cat_oof_pred_pro,cat_test_pro=fit_and_predict(model=cat_model,num_folds=5,name='cat')
print(f"cat_test_pro[:10]:{cat_test_pro[:10]}")

#Trial 35 finished with value: 0.1693021894626871 and parameters: 
xgb_params = {'random_state': 2025, 'n_estimators': 800, 
 'learning_rate': 0.009826644028525231, 'max_depth': 10,
 'reg_alpha': 0.08277318651348423, 'reg_lambda': 0.7719612355688399, 
 'subsample': 0.9579828266704034, 'colsample_bytree': 0.6228502853913586, 'min_child_weight': 3,
  'scale_pos_weight': 2.5,'tree_method':'gpu_hist',
 'objective': 'binary:logistic',
}
xgb_model = XGBClassifier(**xgb_params)

xgb_oof_pred_pro,xgb_test_pro=fit_and_predict(model=xgb_model,num_folds=20,name='xgb')
print(f"xgb_test_pro[:10]:{xgb_test_pro[:10]}")

## 7.find best fusion weight

In [ ]:
#模型的参数选择
steps=100
best_w1=1
best_w2=1
init_pros=(best_w1*lgb_oof_pred_pro+best_w2*cat_oof_pred_pro+(steps-best_w1-best_w2)*xgb_oof_pred_pro)/steps
best_pauc=pauc_above_tpr(train[Config.TARGET_NAME].values, init_pros)[1]
print(f"start best_pauc:{best_pauc}")

#这里不找最佳参数了,就直接用平均作为预测结果
if len(test)<1:#这个无论是线上还是线下都不运行
    best_w1,best_w2=34,33
else:
    #最少设为1是为了不忽略任何一个模型的预测结果
    for w1 in range(1,steps-2):#lgb的权重
        for w2 in range(1,steps-2):#cat的权重
            w3=steps-w1-w2
            #现在的预测结果
            cur_pros=(w1*lgb_oof_pred_pro+w2*cat_oof_pred_pro+w3*xgb_oof_pred_pro)/steps
            cur_pauc=pauc_above_tpr(train[Config.TARGET_NAME].values, cur_pros)[1]
            if cur_pauc>best_pauc:
                best_w1=w1
                best_w2=w2
                best_pauc=cur_pauc
                print(f"best_w1:{best_w1},best_w2:{best_w2},best_pauc:{best_pauc}")


best_pros=(best_w1*lgb_oof_pred_pro+best_w2*cat_oof_pred_pro+(steps-best_w1-best_w2)*xgb_oof_pred_pro)/steps
best_pauc=pauc_above_tpr(train[Config.TARGET_NAME].values, best_pros)[1]
print(f"end best_pauc:{best_pauc}")
test_pros=(best_w1*lgb_test_pro+best_w2*cat_test_pro+(steps-best_w1-best_w2)*xgb_test_pro)/steps
test['target']=test_pros
print(test_pros[:10])

## 8.Submission

In [ ]:
submission=test[['isic_id','target']]
submission.to_csv("submission.csv",index=None)
submission.head()